# Preprocess for the reference cell type

In [ ]:
import pandas as pd
import scanpy as sc

In [ ]:
celltypist = sc.read_h5ad('../celltypist/raw_data/global/CountAdded_PIP_global_object_for_cellxgene.h5ad')
celltypist

In [ ]:
cellt = pd.DataFrame(celltypist.obs['Predicted_labels_CellTypist'].value_counts())
cellt

In [ ]:
cell = pd.read_csv('./origin/cellType_level.csv')
cell = cell.applymap(lambda x: x.rstrip() if isinstance(x, str) else x)
cell

In [ ]:
ss = pd.DataFrame(set(cellt.index.tolist()) & set(cell['level_3'].tolist()))
ss.columns = ['celltype']

In [ ]:
ss

In [ ]:
cells = pd.merge(cell, ss, left_on='level_3', right_on='celltype')
cells = cells[['level_1','level_2','level_3']]
cells

In [ ]:
cells['level_1'].value_counts()

In [ ]:
oth = pd.DataFrame(set(cellt.index.tolist()) - set(ss['celltype'].tolist()))
oth.columns = ['level_3']
oth['level_2'] = ['B cells','B cells','T cells','T cells','T cells','T cells']
oth['level_1'] = ['Lymphoid']*6

In [ ]:
oth = oth[['level_1','level_2','level_3']]
oth

In [ ]:
ot = pd.DataFrame(set(cell['level_3'].tolist()) - set(cellt.index.tolist()))
ot

In [ ]:
cell_a = pd.concat([cell, oth])
cell_a

In [ ]:
cell_a.to_csv('./origin/sc_105.csv', index=None)

# Collate the results of eight algorithms

In [ ]:
import pandas as pd
import scanpy as sc
import os

## singler

In [ ]:
part = pd.read_csv('./singler/predicted_celltype_2.csv', index_col=0)
part

## scpred

In [ ]:
folder_path = "./scpred/"
dataframes = []

In [ ]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path,index_col=0)
        dataframes.append(df)

In [ ]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [ ]:
merged_df.columns = ['cellname', 'celltype']

In [ ]:
merged_df.index = merged_df['cellname'].tolist()

In [ ]:
merged_df

In [ ]:
merged_df['celltype'].value_counts()

In [ ]:
merged_df.to_csv('scpred_all.csv')

In [ ]:
data = pd.read_csv('/path/to/annotation_project/cell_annotation/scPred/celltype_summary/combined_data.csv')
data

In [ ]:
data = pd.read_csv('/path/to/annotation_project/cell_annotation/scPred/cell_summary_glm/combined_data.csv')
data

In [ ]:
dat = data[['X','scpred_no_rejection']]
dat

In [ ]:
dat.columns = ['cellname', 'celltype']

In [ ]:
dat.index = dat['cellname'].tolist()
dat

In [ ]:
dat.to_csv('scpred_all.csv')

## ACTINN

In [ ]:
folder_path = "actinn/"
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()][0:25]
print(subfolders)

In [ ]:
i = 0
cell = pd.read_csv(subfolders[i]+'/predicted_label.txt', sep='\t')
cell.index = cell['cellname'].tolist()
pred = pd.read_csv(subfolders[i]+'/predicted_probabilities.txt', sep='\t', index_col=0).T
cell_pred = pd.concat([cell,pred], axis=1)
cell_pred

In [ ]:
cell_pred[['cellname','celltype']].to_csv('actinn_part1.csv')

In [ ]:
for i in range(1,25):
    print(i)
    cell = pd.read_csv(subfolders[i]+'/predicted_label.txt', sep='\t')
    cell.index = cell['cellname'].tolist()
    pred = pd.read_csv(subfolders[i]+'/predicted_probabilities.txt', sep='\t', index_col=0).T
    cell_pre = pd.concat([cell,pred], axis=1)
    cell_pred = pd.concat([cell_pred,cell_pre], axis=0)

In [ ]:
cell_pred

In [ ]:
cell_pred['celltype'].value_counts()

In [ ]:
cell_pred[['cellname','celltype']].to_csv('actinn/actinn_all.csv')

## scmap

In [ ]:
# part_1
# 2000 genes, 12.3% NA
# 1000 genes, 12.1% NA
# 500 genes, 9.9%

In [ ]:
folder_path = "./scmap/"
dataframes = []

In [ ]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path)
        dataframes.append(df)

In [ ]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [ ]:
merged_df.columns = ['cellname', 'celltype', 'similarity']
merged_df.index = merged_df['cellname'].tolist()

In [ ]:
merged_df

In [ ]:
pd.DataFrame(merged_df['celltype'].value_counts())

In [ ]:
pd.DataFrame(merged_df['similarity'].isna())['similarity'].value_counts()

In [ ]:
merged_df[['cellname', 'celltype']].to_csv('scmap_all.csv')

In [ ]:
part_1 = pd.read_csv('../actinn/query/part_1/barcodes.tsv', header=None)
part_1

In [ ]:
merged_df_part1 = merged_df.loc[part_1[0].tolist(),:]
merged_df_part1

In [ ]:
merged_df_part1[['cellname', 'celltype']].to_csv('scmap_part1.csv')

## scarch

In [ ]:
cell = pd.read_csv('scarch/part_1/preds.csv',index_col=0)
id = pd.read_csv('actinn/part_1/barcodes.tsv', header=None)
cell.index = id[0].tolist()
cell.columns = ['celltype']
cell['cellname'] = id[0].tolist()
cell

In [ ]:
cell['celltype'].value_counts()

In [ ]:
cell[['cellname', 'celltype']].to_csv('scarch_part1.csv')

In [ ]:
prefix = "part_"

In [ ]:
for i in range(2,26):
    print(i)
    folder_name = f"{prefix}{i}"
    cel = pd.read_csv('scarch/'+folder_name+'/preds.csv',index_col=0)
    id = pd.read_csv('actinn/'+folder_name+'/barcodes.tsv', header=None)
    cel.index = id[0].tolist()
    cel.columns = ['celltype']
    cel['cellname'] = id[0].tolist()
    cell = pd.concat([cell,cel], axis=0)

In [ ]:
cell

In [ ]:
cell[['cellname', 'celltype']].to_csv('scarch_all.csv')

## chetah unassigned for the most

In [ ]:
data = pd.read_csv('chetah/predicted_celltype_1.csv',index_col=0)
data

In [ ]:
data.columns = ['cellname', 'celltype']

In [ ]:
data[['cellname', 'celltype']].to_csv('chetah_part1.csv')

In [ ]:
folder_path = "./chetah/"
dataframes = []

In [ ]:
for file_name in os.listdir(folder_path):
    if file_name.endswith(".csv"):
        file_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(file_path,index_col=0)
        dataframes.append(df)

In [ ]:
merged_df = pd.concat(dataframes, axis=0, ignore_index=True)

In [ ]:
merged_df

In [ ]:
merged_df.columns = ['cellname', 'celltype']
merged_df.index = merged_df['cellname'].tolist()

In [ ]:
merged_df.to_csv('chetah_all.csv')

In [ ]:
data = pd.read_csv('./chetah_all.csv')
data

In [ ]:
data['celltype'].value_counts()

## singcellnet

In [ ]:
data = pd.read_csv('./singcellnet/combined.csv', index_col=0)
data

In [ ]:
data.index = data['Cell'].tolist()
data

In [ ]:
others = pd.read_csv('./all_method4.csv', index_col=0)
others

In [ ]:
len(set(data.index.tolist()) & set(others.index.tolist()))

In [ ]:
data.to_csv('singcellnet_all.csv')

## test

In [ ]:
rna = sc.read_h5ad('../6_integrate/processed/adata_rna.h5ad')
rna

In [ ]:
actinn = pd.read_csv('actinn_all.csv', index_col=0)
actinn  

In [ ]:
scamp = pd.read_csv('scmap_all.csv', index_col=0)
scamp

In [ ]:
print(set(actinn.index.tolist()) == set(rna.obs.index.tolist()))

# Majority voting

In [ ]:
import pandas as pd
import scanpy as sc

## merge data

In [ ]:
all = pd.read_csv('./new_data/combined_26.csv', index_col=0)
all

In [ ]:
all.columns = ['chetah', 'scmap', 'singlecellnet', 'scpred', 'singler']
all

In [ ]:
celltypist = pd.read_csv('./new_data/celltypist.csv',index_col=0)
celltypist

In [ ]:
all['celltypist'] = celltypist['predicted_labels'].tolist()
all

In [ ]:
actinn = pd.read_csv('./new_data/actinn.txt', sep='\t')
actinn

In [ ]:
all['actinn'] = actinn['celltype'].tolist()
all

In [ ]:
scarch = pd.read_csv('./new_data/scarch.csv')
scarch

In [ ]:
all['scarch'] = scarch['0'].tolist()
all

In [ ]:
all.to_csv('./method8_level3.csv')

## level_2 data

In [ ]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

In [ ]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

In [ ]:
all_2 = all.replace(result)
all_2

In [ ]:
all_2.to_csv('./method8_level2.csv')

## preprocess for cell type

In [ ]:
all = pd.read_csv('./scarch_all.csv', index_col=0)[['celltype']]
all.columns = ['scarch']
actinn = pd.read_csv('./actinn_all.csv', index_col=0)
all['actinn'] = actinn['celltype'].tolist()

In [ ]:
all

In [ ]:
actinn.index.tolist() == all.index.tolist()

In [ ]:
celltypist = pd.read_csv('../6_integrate/processed/celltypist/cellTypist.csv', index_col=0)
celltypist = celltypist.reindex(all.index.tolist())

In [ ]:
celltypist.index.tolist() == all.index.tolist()

In [ ]:
all['celltypist'] = celltypist['predicted_labels'].tolist()

In [ ]:
all

In [ ]:
scmap = pd.read_csv('./scmap_all.csv', index_col=0)
scmap = scmap.reindex(all.index.tolist())
all['scmap'] = scmap['celltype'].tolist()

In [ ]:
scmap.index.tolist() == all.index.tolist()

In [ ]:
singcellnet = pd.read_csv('./singcellnet_all.csv', index_col=0)
singcellnet

In [ ]:
singcellnet.index.tolist() == all.index.tolist()

In [ ]:
all['singcellnet'] = singcellnet['CellType'].tolist()

In [ ]:
all

In [ ]:
scpred = pd.read_csv('./scpred_all.csv', index_col=0)
scpred = scpred.reindex(all.index.tolist())

In [ ]:
scpred.index.tolist() == all.index.tolist()

In [ ]:
scpred

In [ ]:
all['scpred'] = scpred['celltype'].tolist()

In [ ]:
all

In [ ]:
all.to_csv('./all_method6.csv')

In [ ]:
all = pd.read_csv('./all_method6.csv', index_col=0)
all

In [ ]:
sr = pd.read_csv('/path/to/annotation_project/cell_annotation/combined_1-25/sr_combined_data-1-25.csv', index_col=2)
sr

In [ ]:
sr.index.tolist() == all.index.tolist()

In [ ]:
sr = sr.reindex(all.index.tolist())

In [ ]:
sr.index.tolist() == all.index.tolist()

In [ ]:
all['singler'] = sr['V2'].tolist()
all

In [ ]:
chetah = pd.read_csv('./huizong/chetah_all.csv',index_col=0)
chetah

In [ ]:
chetah.index.tolist() == all.index.tolist()

In [ ]:
chetah = chetah.reindex(all.index.tolist())

In [ ]:
chetah.index.tolist() == all.index.tolist()

In [ ]:
all['chetah'] = chetah['celltype'].tolist()
all

In [ ]:
del all['mv_3']

In [ ]:
all.to_csv('./all_method8.csv')

## preprocess for cell type level2

In [ ]:
all = pd.read_csv('./all_method8.csv', index_col=0)
all

In [ ]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

In [ ]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

In [ ]:
all_2 = all.replace(result)
all_2

In [ ]:
all_2.to_csv('all_method8_level2.csv')

## compute

In [ ]:
import pandas as pd

# Vectorized implementation of majority voting
def majority_vote_vectorized(df):
    # Count the frequency of each value per row
    mode_counts = df.apply(lambda row: row.value_counts(), axis=1).fillna(0)

    # Find the most frequent value and its occurrence count
    max_counts = mode_counts.max(axis=1)
    majority_values = mode_counts.idxmax(axis=1)

    # Determine validity based on the occurrence count (threshold >= 4)
    result = majority_values.where(max_counts >= 4, "notAvailable")
    return result

all = pd.read_csv('./all_method6.csv', index_col=0)
all['mv_3'] = majority_vote_vectorized(all)
all.to_csv('./all_method6.csv')

all = pd.read_csv('./all_method6_level2.csv', index_col=0)
all['mv_3'] = majority_vote_vectorized(all)
all.to_csv('./all_method6_level2.csv')

# Process for the new datasets

In [ ]:
import pandas as pd
import scanpy as sc

## merge

In [ ]:
all = pd.read_csv('./new_data/combined_26.csv', index_col=0)
all

In [ ]:
all.columns = ['chetah', 'scmap', 'singlecellnet', 'scpred', 'singler']
all

In [ ]:
celltypist = pd.read_csv('./new_data/celltypist.csv',index_col=0)
celltypist

In [ ]:
all['celltypist'] = celltypist['predicted_labels'].tolist()
all

In [ ]:
actinn = pd.read_csv('./new_data/actinn.txt', sep='\t')
actinn

In [ ]:
all['actinn'] = actinn['celltype'].tolist()
all

In [ ]:
scarch = pd.read_csv('./new_data/scarch.csv')
scarch

In [ ]:
all['scarch'] = scarch['0'].tolist()
all

In [ ]:
all.to_csv('./method8_level3.csv')

## merge level_2 data

In [ ]:
cell = pd.read_csv('./origin/sc_105.csv')
cell

In [ ]:
result = dict(zip(cell['level_3'], cell['level_2']))
result

In [ ]:
all_2 = all.replace(result)
all_2

In [ ]:
all_2.to_csv('./method8_level2.csv')